In [10]:
import sys
import os
import pandas as pd
sys.path.append("../")
sys.path.append("../../")

In [11]:
from scale_rl.envs.dmc import DMC_HARD

def replace_underbar_to_hypen(env_name_list):
    for idx in range(len(env_name_list)):
        env_name_list[idx] = env_name_list[idx].replace('_', '-')
    return env_name_list
DMC_HARD = replace_underbar_to_hypen(DMC_HARD)

In [12]:
eval_df = pd.read_csv('../results/raw/iqrl/iqrl-neat.csv')

In [13]:
eval_df

,env_step,episode_reward,env,seed,agent
0,100000.0,53.545047,dog-run,1,iQRL
1,200000.0,103.467921,dog-run,1,iQRL
2,300000.0,170.103271,dog-run,1,iQRL
3,400000.0,215.778951,dog-run,1,iQRL
4,500000.0,254.356482,dog-run,1,iQRL
...,...,...,...,...,...
625,2600000.0,883.150806,humanoid-walk,3,iQRL
626,2700000.0,885.483124,humanoid-walk,3,iQRL
627,2800000.0,900.699548,humanoid-walk,3,iQRL
628,2900000.0,884.092828,humanoid-walk,3,iQRL


In [14]:
eval_df.rename(columns = {'env' : 'env_name', 'episode_reward' : 'value', 'agent': 'exp_name'}, inplace = True)
eval_df

,env_step,value,env_name,seed,exp_name
0,100000.0,53.545047,dog-run,1,iQRL
1,200000.0,103.467921,dog-run,1,iQRL
2,300000.0,170.103271,dog-run,1,iQRL
3,400000.0,215.778951,dog-run,1,iQRL
4,500000.0,254.356482,dog-run,1,iQRL
...,...,...,...,...,...
625,2600000.0,883.150806,humanoid-walk,3,iQRL
626,2700000.0,885.483124,humanoid-walk,3,iQRL
627,2800000.0,900.699548,humanoid-walk,3,iQRL
628,2900000.0,884.092828,humanoid-walk,3,iQRL


In [15]:
eval_df["env_step"] = eval_df["env_step"].astype({'env_step' :'int'})
eval_df

,env_step,value,env_name,seed,exp_name
0,100000,53.545047,dog-run,1,iQRL
1,200000,103.467921,dog-run,1,iQRL
2,300000,170.103271,dog-run,1,iQRL
3,400000,215.778951,dog-run,1,iQRL
4,500000,254.356482,dog-run,1,iQRL
...,...,...,...,...,...
625,2600000,883.150806,humanoid-walk,3,iQRL
626,2700000,885.483124,humanoid-walk,3,iQRL
627,2800000,900.699548,humanoid-walk,3,iQRL
628,2900000,884.092828,humanoid-walk,3,iQRL


In [16]:
eval_df["metric"] = "avg_return"
eval_df

,env_step,value,env_name,seed,exp_name,metric
0,100000,53.545047,dog-run,1,iQRL,avg_return
1,200000,103.467921,dog-run,1,iQRL,avg_return
2,300000,170.103271,dog-run,1,iQRL,avg_return
3,400000,215.778951,dog-run,1,iQRL,avg_return
4,500000,254.356482,dog-run,1,iQRL,avg_return
...,...,...,...,...,...,...
625,2600000,883.150806,humanoid-walk,3,iQRL,avg_return
626,2700000,885.483124,humanoid-walk,3,iQRL,avg_return
627,2800000,900.699548,humanoid-walk,3,iQRL,avg_return
628,2900000,884.092828,humanoid-walk,3,iQRL,avg_return


In [17]:
sort = ['exp_name', 'env_name', 'seed', 'metric', 'env_step', 'value']
eval_df = eval_df[sort]
eval_df

,exp_name,env_name,seed,metric,env_step,value
0,iQRL,dog-run,1,avg_return,100000,53.545047
1,iQRL,dog-run,1,avg_return,200000,103.467921
2,iQRL,dog-run,1,avg_return,300000,170.103271
3,iQRL,dog-run,1,avg_return,400000,215.778951
4,iQRL,dog-run,1,avg_return,500000,254.356482
...,...,...,...,...,...,...
625,iQRL,humanoid-walk,3,avg_return,2600000,883.150806
626,iQRL,humanoid-walk,3,avg_return,2700000,885.483124
627,iQRL,humanoid-walk,3,avg_return,2800000,900.699548
628,iQRL,humanoid-walk,3,avg_return,2900000,884.092828


In [18]:
eval_df.to_csv("../results/0118_iQRL_dmc_hard.csv")

In [19]:
DMC_STEPS = 1e6 # 1M

In [20]:
overall_df = [] 

for env_name in DMC_HARD:
    env_df = eval_df[eval_df["env_name"] == env_name]
    if len(env_df) == 0:
        continue
    # assert max(env_df["env_step"]) == HB_STEPS
    env_df = env_df[env_df["env_step"] == DMC_STEPS]
    env_df["value"] = env_df["value"] / 1000.0
    num_seeds = env_df["seed"].nunique()

    grouped_data = env_df.groupby("env_step")["value"]

    env_steps = grouped_data.mean().index.values
    mean = float(grouped_data.mean().values)
    std_err = float(grouped_data.sem().values)

    print(f"{env_name} - number of seeds: {num_seeds}")
    print(f"{round(mean, 3)} ± {round(1.96 * std_err, 3)}")
    
    overall_df.append(env_df)
    
overall_df = pd.concat(overall_df)
mean = overall_df["value"].mean()
std_err = overall_df["value"].sem()
print(f"Overall: {round(mean, 3)} ± {round(1.96 * std_err, 3)}")

humanoid-stand - number of seeds: 3
0.728 ± 0.072
humanoid-walk - number of seeds: 3
0.689 ± 0.046
humanoid-run - number of seeds: 3
0.189 ± 0.021
dog-stand - number of seeds: 3
0.927 ± 0.029
dog-walk - number of seeds: 3
0.867 ± 0.039
dog-run - number of seeds: 3
0.38 ± 0.044
dog-trot - number of seeds: 3
0.713 ± 0.197
Overall: 0.642 ± 0.111
